In [1]:
import pandas as pd

# Load data
deliveries = pd.read_csv('../data/deliveries.csv')
matches = pd.read_csv('../data/matches.csv')

print("Deliveries shape:", deliveries.shape)
print("Deliveries columns:", deliveries.columns.tolist())
print("\nMatches shape:", matches.shape)
print("Matches columns:", matches.columns.tolist())

# Define target variable (winner for matches data)
TARGET = 'winner'

# Select relevant columns and prepare data
DROP = ['id', 'season', 'city', 'date', 'toss_winner', 'toss_decision', 'dl_applied', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']

# Prepare features and target
X = matches.drop(columns=[col for col in DROP + [TARGET] if col in matches.columns])
y = matches[TARGET]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)

Deliveries shape: (150460, 21)
Deliveries columns: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs', 'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs', 'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed', 'dismissal_kind', 'fielder']

Matches shape: (636, 18)
Matches columns: ['id', 'season', 'city', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']

Features shape: (636, 5)
Target shape: (636,)


In [2]:
data = deliveries.merge(matches, left_on='match_id', right_on='id')

print(data.shape)
data.head()

(150460, 39)


,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN


In [4]:
# FIRST create is_wicket
data['is_wicket'] = data['player_dismissed'].notnull().astype(int)

# THEN groupby
over_data = data.groupby(['match_id', 'inning', 'over']).agg({
    'total_runs': 'sum',
    'extra_runs': 'sum',
    'is_wicket': 'sum'
}).reset_index()

In [5]:
print(over_data.shape)
over_data.head()


(24388, 6)


,match_id,inning,over,total_runs,extra_runs,is_wicket
0,1,1,1,7,3,0
1,1,1,2,16,1,1
2,1,1,3,6,0,0
3,1,1,4,4,0,0
4,1,1,5,9,0,0


In [6]:
over_data['cum_runs'] = over_data.groupby(['match_id', 'inning'])['total_runs'].cumsum()

over_data['cum_wickets'] = over_data.groupby(['match_id', 'inning'])['is_wicket'].cumsum()

over_data.head()

,match_id,inning,over,total_runs,extra_runs,is_wicket,cum_runs,cum_wickets
0,1,1,1,7,3,0,7,0
1,1,1,2,16,1,1,23,1
2,1,1,3,6,0,0,29,1
3,1,1,4,4,0,0,33,1
4,1,1,5,9,0,0,42,1


In [ ]:
# NEW FEATURES (To improve accuracy)

# run rate
over_data['run_rate'] = over_data['cum_runs'] / over_data['over']

# wickets remaining
over_data['wickets_remaining'] = 10 - over_data['cum_wickets']

# avoid NaN (division issue)
over_data['run_rate'] = over_data['run_rate'].fillna(0)

In [7]:
team_data = matches[['id', 'team1', 'winner']]

over_data = over_data.merge(team_data, left_on='match_id', right_on='id')

over_data.head()

,match_id,inning,over,total_runs,extra_runs,is_wicket,cum_runs,cum_wickets,id,team1,winner
0,1,1,1,7,3,0,7,0,1,Sunrisers Hyderabad,Sunrisers Hyderabad
1,1,1,2,16,1,1,23,1,1,Sunrisers Hyderabad,Sunrisers Hyderabad
2,1,1,3,6,0,0,29,1,1,Sunrisers Hyderabad,Sunrisers Hyderabad
3,1,1,4,4,0,0,33,1,1,Sunrisers Hyderabad,Sunrisers Hyderabad
4,1,1,5,9,0,0,42,1,1,Sunrisers Hyderabad,Sunrisers Hyderabad


In [8]:
over_data['team1_won'] = (over_data['winner'] == over_data['team1']).astype(int)

In [9]:
over_data = over_data.drop(columns=['id', 'team1', 'winner'])

In [10]:
over_data.head()

,match_id,inning,over,total_runs,extra_runs,is_wicket,cum_runs,cum_wickets,team1_won
0,1,1,1,7,3,0,7,0,1
1,1,1,2,16,1,1,23,1,1
2,1,1,3,6,0,0,29,1,1
3,1,1,4,4,0,0,33,1,1
4,1,1,5,9,0,0,42,1,1


In [11]:
X = over_data.drop(columns=['team1_won'])
y = over_data['team1_won']

print(X.shape)
print(y.shape)

(24388, 8)
(24388,)


In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape)
print(X_test.shape)

(19510, 8)
(4878, 8)


In [13]:
X = over_data.select_dtypes(include=['number']).drop(columns=['team1_won'])
y = over_data['team1_won']

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape)
print(X_test.shape)

(19510, 8)
(4878, 8)


In [5]:
from sklearn.tree import DecisionTreeClassifier

#model = DecisionTreeClassifier(max_depth=6, random_state=42)
model = DecisionTreeClassifier(
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current n

In [6]:
y_pred = model.predict(X_test)

In [7]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7004920049200491


In [8]:
# training predictions
y_train_pred = model.predict(X_train)

from sklearn.metrics import accuracy_score

print("Training Accuracy:", accuracy_score(y_train, y_train_pred))

Training Accuracy: 0.7375192209123527
